In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import FloatSlider, HTML, HBox, Layout
from IPython.display import display

# ============================================================
# PARALLEL IMPLEMENTATION — PARTIAL-FRACTION EXPANSION
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12.5,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.par-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.par-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.par-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.48;
    margin-bottom:7px;
}

.par-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:6px;
    font-size:13.5px;
    line-height:1.45;
}

.par-result{
    background:#fff8e6;
    border:1px solid #d8b451;
}

.par-title{
    color:#0d47a1;
    font-weight:bold;
    font-size:14.5px;
    margin-bottom:5px;
}

.par-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:6px 0;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="par-root">

<div class="par-header">
Parallel Implementation — Partial-Fraction Expansion
</div>

<div class="par-doc">

In a parallel implementation, the transfer function is expressed as a
<b>sum</b> of simpler transfer functions rather than as a product.

For the sixth-order filter used in this notebook, the decomposition has the
form

<div class="par-equation">
<b>
H(z) = C₀ + H₁(z) + H₂(z) + H₃(z).
</b>
</div>

The same input signal is applied simultaneously to every branch,

<div class="par-equation">
x[n] → {C₀, H₁(z), H₂(z), H₃(z)}
</div>

and the branch outputs are finally added:

<div class="par-equation">
<b>
y[n] = y₀[n] + y₁[n] + y₂[n] + y₃[n].
</b>
</div>

The slider changes the frequency of the sinusoidal component of the input.
This allows us to observe how the contribution of each parallel branch changes
while the final parallel output remains identical to the direct-form output.

</div>

</div>
"""))

# ============================================================
# FIXED SIXTH-ORDER IIR FILTER
# ============================================================

ORDER = 6
CUTOFF = 0.35

b,a = signal.butter(ORDER,CUTOFF,btype='low',output='ba')

# ============================================================
# PARTIAL-FRACTION EXPANSION
# ============================================================

residues,poles,direct_terms = signal.residuez(b,a)

C0 = float(np.real_if_close(direct_terms[0]))

# ============================================================
# COMBINE COMPLEX-CONJUGATE TERMS INTO REAL SECOND-ORDER BRANCHES
# ============================================================

branches = []

used = np.zeros(len(poles),dtype=bool)

for i in range(len(poles)):

    if used[i]:
        continue

    distances = np.abs(poles-np.conj(poles[i]))

    distances[used] = np.inf

    j = np.argmin(distances)

    ri = residues[i]
    rj = residues[j]

    pi = poles[i]
    pj = poles[j]

    b0 = np.real(ri+rj)

    b1 = -np.real(ri*pj+rj*pi)

    a0 = 1.0

    a1 = -np.real(pi+pj)

    a2 = np.real(pi*pj)

    branch_b = np.array([b0,b1])

    branch_a = np.array([a0,a1,a2])

    branches.append((branch_b,branch_a))

    used[i] = True

    used[j] = True

NUM_BRANCHES = len(branches)

# ============================================================
# FREQUENCY RESPONSES
# ============================================================

omega = np.linspace(0.0,np.pi,1024)

_,H_direct = signal.freqz(b,a,worN=omega)

branch_H = []

for branch_b,branch_a in branches:

    _,Hk = signal.freqz(branch_b,branch_a,worN=omega)

    branch_H.append(Hk)

branch_H = np.array(branch_H)

H_parallel = np.full_like(H_direct,C0,dtype=complex)

for Hk in branch_H:

    H_parallel += Hk

# ============================================================
# INPUT SIGNAL
# ============================================================

N = 80

n = np.arange(N)

frequency_slider = FloatSlider(value=0.22,min=0.05,max=0.95,step=0.01,description='ω₀/π:',continuous_update=True,readout_format='.2f',style={'description_width':'45px'},layout=Layout(width='330px'))

controls = HBox([frequency_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='6px 9px',margin='0 0 3px 0'))

# ============================================================
# INPUT GENERATION
# ============================================================

def create_input(freq):

    x = np.zeros(N)

    x[0] = 1.0

    x += 0.30*np.sin(freq*np.pi*n)

    return x

# ============================================================
# PARALLEL IMPLEMENTATION
# ============================================================

def calculate_parallel(x):

    outputs = []

    y0 = C0*x

    outputs.append(y0)

    for branch_b,branch_a in branches:

        yk = signal.lfilter(branch_b,branch_a,x)

        outputs.append(yk)

    y_parallel = np.sum(np.array(outputs),axis=0)

    return outputs,y_parallel

# ============================================================
# PRECOMPUTE FIXED TIME-DOMAIN AXIS LIMITS
# ============================================================

all_internal_values = []

all_final_values = []

all_differences = []

frequency_values = np.arange(frequency_slider.min,frequency_slider.max+frequency_slider.step/2,frequency_slider.step)

for freq in frequency_values:

    x_test = create_input(freq)

    outputs_test,y_parallel_test = calculate_parallel(x_test)

    y_direct_test = signal.lfilter(b,a,x_test)

    for yk in outputs_test:

        all_internal_values.extend(yk)

    all_final_values.extend(y_parallel_test)

    all_final_values.extend(y_direct_test)

    all_differences.extend(y_direct_test-y_parallel_test)

internal_min = np.min(all_internal_values)

internal_max = np.max(all_internal_values)

internal_margin = 0.08*(internal_max-internal_min)

INTERNAL_YMIN = internal_min-internal_margin

INTERNAL_YMAX = internal_max+internal_margin

final_min = np.min(all_final_values)

final_max = np.max(all_final_values)

final_margin = 0.08*(final_max-final_min)

FINAL_YMIN = final_min-final_margin

FINAL_YMAX = final_max+final_margin

difference_limit = max(1e-14,1.20*np.max(np.abs(all_differences)))

# ============================================================
# INITIAL VALUES
# ============================================================

current_frequency = frequency_slider.value

x = create_input(current_frequency)

branch_outputs,y_parallel = calculate_parallel(x)

y_direct = signal.lfilter(b,a,x)

difference = y_direct-y_parallel

# ============================================================
# INFORMATION BOX
# ============================================================

result_html = HTML(layout=Layout(width=CONTENT_WIDTH))

# ============================================================
# FIGURE 1 — PARALLEL STRUCTURE
# CREATED ONCE
# ============================================================

fig1,ax_structure = plt.subplots(figsize=(9.0,3.0))

fig1.canvas.toolbar_visible = False
fig1.canvas.header_visible = False
fig1.canvas.footer_visible = False

ax_structure.set_xlim(0,10)

ax_structure.set_ylim(0,5)

ax_structure.axis('off')

ax_structure.set_title('Parallel Structure of the Sixth-Order Filter')

# Input vertical bus
ax_structure.plot([1.2,1.2],[0.65,4.35],linewidth=1.3)

ax_structure.annotate('',xy=(1.2,4.55),xytext=(1.2,4.35),arrowprops={'arrowstyle':'->','linewidth':1.4})

ax_structure.text(1.2,4.72,r'$x[n]$',ha='center',fontsize=11,fontweight='bold')

# Output vertical bus
ax_structure.plot([8.8,8.8],[0.65,4.35],linewidth=1.3)

ax_structure.annotate('',xy=(8.8,4.55),xytext=(8.8,4.35),arrowprops={'arrowstyle':'->','linewidth':1.4})

ax_structure.text(8.8,4.72,r'$y[n]$',ha='center',fontsize=11,fontweight='bold')

branch_y_positions = [3.8,2.9,2.0,1.1]

branch_labels = [r'$C_0$']+[rf'$H_{k+1}(z)$' for k in range(NUM_BRANCHES)]

for y_pos,label in zip(branch_y_positions,branch_labels):

    ax_structure.annotate('',xy=(3.0,y_pos),xytext=(1.2,y_pos),arrowprops={'arrowstyle':'->','linewidth':1.3})

    rect = plt.Rectangle((3.0,y_pos-0.30),2.5,0.60,fill=False,linewidth=1.3)

    ax_structure.add_patch(rect)

    ax_structure.text(4.25,y_pos,label,ha='center',va='center',fontsize=11,fontweight='bold')

    ax_structure.annotate('',xy=(8.8,y_pos),xytext=(5.5,y_pos),arrowprops={'arrowstyle':'->','linewidth':1.3})

# Summation indication
ax_structure.text(8.8,0.35,r'$\sum$',ha='center',va='center',fontsize=18,fontweight='bold')

plt.subplots_adjust(left=0.03,right=0.98,top=0.86,bottom=0.05)

# ============================================================
# FIGURE 2 — FREQUENCY RESPONSES
# CREATED ONCE
# ============================================================

fig2,(ax_branches,ax_total) = plt.subplots(1,2,figsize=(9.0,3.6))

fig2.canvas.toolbar_visible = False
fig2.canvas.header_visible = False
fig2.canvas.footer_visible = False

# ------------------------------------------------------------
# INDIVIDUAL PARALLEL BRANCH RESPONSES
# ------------------------------------------------------------

for k,Hk in enumerate(branch_H):

    magnitude = 20*np.log10(np.maximum(np.abs(Hk),1e-12))

    ax_branches.plot(omega/np.pi,magnitude,linewidth=1.3,label=f'H{k+1}')

ax_branches.axhline(20*np.log10(abs(C0)),linestyle='--',linewidth=1.0,label='C₀')

ax_branches.set_xlim(0,1)

ax_branches.set_ylim(-80,30)

ax_branches.set_title('Individual Parallel Branch Responses')

ax_branches.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax_branches.set_ylabel('Magnitude (dB)')

ax_branches.grid(True,linestyle=':',alpha=0.30)

ax_branches.legend(loc='lower left')

# ------------------------------------------------------------
# DIRECT VS PARALLEL FREQUENCY RESPONSE
# ------------------------------------------------------------

direct_mag = 20*np.log10(np.maximum(np.abs(H_direct),1e-12))

parallel_mag = 20*np.log10(np.maximum(np.abs(H_parallel),1e-12))

ax_total.plot(omega/np.pi,direct_mag,linewidth=1.5,label='Direct implementation')

ax_total.plot(omega/np.pi,parallel_mag,'--',linewidth=1.3,label='Parallel implementation')

ax_total.set_xlim(0,1)

ax_total.set_ylim(-160,5)

ax_total.set_title('Overall Frequency Response')

ax_total.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax_total.set_ylabel('Magnitude (dB)')

ax_total.grid(True,linestyle=':',alpha=0.30)

ax_total.legend(loc='lower left')

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# FIGURE 3 — PARALLEL BRANCH OUTPUTS
# CREATED ONCE
# ============================================================

fig3,ax_internal = plt.subplots(figsize=(9.0,3.5))

fig3.canvas.toolbar_visible = False
fig3.canvas.header_visible = False
fig3.canvas.footer_visible = False

internal_lines = []

for k,yk in enumerate(branch_outputs):

    label = 'C₀ branch' if k == 0 else f'Branch H{k}'

    line, = ax_internal.plot(n,yk,linewidth=1.2,label=label)

    internal_lines.append(line)

ax_internal.set_xlim(0,N-1)

ax_internal.set_ylim(INTERNAL_YMIN,INTERNAL_YMAX)

ax_internal.set_title('Internal Parallel Branch Outputs')

ax_internal.set_xlabel('Sample index n')

ax_internal.set_ylabel('Branch output')

ax_internal.grid(True,linestyle=':',alpha=0.30)

ax_internal.legend(loc='upper right',ncol=2)

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16)

# ============================================================
# FIGURE 4 — FINAL OUTPUT COMPARISON
# CREATED ONCE
# ============================================================

fig4,(ax_output,ax_difference) = plt.subplots(1,2,figsize=(9.0,3.5))

fig4.canvas.toolbar_visible = False
fig4.canvas.header_visible = False
fig4.canvas.footer_visible = False

# ------------------------------------------------------------
# FINAL OUTPUT
# ------------------------------------------------------------

direct_line, = ax_output.plot(n,y_direct,linewidth=1.5,label='Direct implementation')

parallel_line, = ax_output.plot(n,y_parallel,'--',linewidth=1.3,label='Parallel implementation')

ax_output.set_xlim(0,N-1)

ax_output.set_ylim(FINAL_YMIN,FINAL_YMAX)

ax_output.set_title('Final Output Comparison')

ax_output.set_xlabel('Sample index n')

ax_output.set_ylabel('y[n]')

ax_output.grid(True,linestyle=':',alpha=0.30)

ax_output.legend(loc='upper right')

# ------------------------------------------------------------
# DIFFERENCE
# ------------------------------------------------------------

difference_line, = ax_difference.plot(n,difference,linewidth=1.2)

ax_difference.axhline(0,linewidth=0.8)

ax_difference.set_xlim(0,N-1)

ax_difference.set_ylim(-difference_limit,difference_limit)

ax_difference.set_title('Numerical Difference')

ax_difference.set_xlabel('Sample index n')

ax_difference.set_ylabel(r'$y_D[n]-y_P[n]$')

ax_difference.grid(True,linestyle=':',alpha=0.30)

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.16,wspace=0.28)

# ============================================================
# UPDATE CALLBACK
# ============================================================

def update(change=None):

    frequency = frequency_slider.value

    # --------------------------------------------------------
    # NEW INPUT SIGNAL
    # --------------------------------------------------------

    x_new = create_input(frequency)

    # --------------------------------------------------------
    # DIRECT IMPLEMENTATION
    # --------------------------------------------------------

    y_direct_new = signal.lfilter(b,a,x_new)

    # --------------------------------------------------------
    # PARALLEL IMPLEMENTATION
    # --------------------------------------------------------

    outputs_new,y_parallel_new = calculate_parallel(x_new)

    difference_new = y_direct_new-y_parallel_new

    # --------------------------------------------------------
    # UPDATE INTERNAL BRANCH OUTPUTS
    # --------------------------------------------------------

    for k in range(len(outputs_new)):

        internal_lines[k].set_ydata(outputs_new[k])

    # --------------------------------------------------------
    # UPDATE FINAL OUTPUTS
    # --------------------------------------------------------

    direct_line.set_ydata(y_direct_new)

    parallel_line.set_ydata(y_parallel_new)

    difference_line.set_ydata(difference_new)

    # --------------------------------------------------------
    # NUMERICAL INFORMATION
    # --------------------------------------------------------

    maximum_difference = np.max(np.abs(difference_new))

    rms_values = [np.sqrt(np.mean(yk**2)) for yk in outputs_new]

    rows = f"""
    <tr>
    <td style="padding:2px 10px;"><b>C₀ branch</b></td>
    <td style="padding:2px 10px;">C₀ = {C0:.6f}</td>
    <td style="padding:2px 10px;">RMS output = {rms_values[0]:.6f}</td>
    </tr>
    """

    for k,(branch_b,branch_a) in enumerate(branches):

        rows += f"""
        <tr>
        <td style="padding:2px 10px;"><b>H<sub>{k+1}</sub>(z)</b></td>
        <td style="padding:2px 10px;">
        b = [{branch_b[0]:.6f}, {branch_b[1]:.6f}]
        </td>
        <td style="padding:2px 10px;">
        RMS output = {rms_values[k+1]:.6f}
        </td>
        </tr>
        """

    result_html.value = f"""
    <div class="par-root">

    <div class="par-box par-result">

    <div class="par-title">
    Current parallel implementation
    </div>

    Input sinusoidal frequency:
    <b>ω₀ = {frequency:.2f}π rad/sample</b>

    &nbsp;&nbsp;&nbsp;

    Maximum |y<sub>direct</sub>[n] − y<sub>parallel</sub>[n]|:
    <b>{maximum_difference:.3e}</b>

    <table style="margin-top:6px;font-size:12.5px;border-collapse:collapse;">
    {rows}
    </table>

    </div>

    </div>
    """

    # --------------------------------------------------------
    # REDRAW EXISTING CANVASES ONLY
    # --------------------------------------------------------

    fig3.canvas.draw_idle()

    fig4.canvas.draw_idle()

# ============================================================
# OBSERVER
# ============================================================

frequency_slider.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(result_html)

display(fig1.canvas)

display(fig2.canvas)

display(controls)

display(fig3.canvas)

display(fig4.canvas)

# ============================================================
# INITIAL UPDATE
# ============================================================

update()